# 01 — Meeting the data

Before any modelling, before any charts, you have to know **what one row means**
in each file. This notebook does that, and nothing else.

It assumes you can read Python but not that you know pandas, machine learning,
or this dataset. Everything is explained as it appears.

**What you'll get out of it:** the seven tables, what a row means in each, why
that matters more than it sounds like it should, and how to read missing values
correctly — including one column where "missing" is the most informative value
in the file.


## Setup

`pandas` is the standard Python library for working with tables. Its main object
is the **DataFrame** — think of it as a spreadsheet you manipulate with code
instead of a mouse: named columns, numbered rows, and operations that apply to
whole columns at once.

The `oulad` imports are this project's own code, in `src/oulad/`.

In [1]:
# Make src/ importable, whichever directory Jupyter was started from.
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "src" / "oulad").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from oulad import load, viz
from oulad import labels as lab

viz.use_project_style()
pd.set_option("display.width", 100)
pd.set_option("display.max_columns", 30)
print("ready")

ready


## One table, to start

`studentInfo` is the natural starting point: it has one row per student
enrolment and it holds the outcome we eventually want to predict.

`.head()` shows the first few rows.

In [2]:
student_info = load.load_table("studentInfo")
student_info.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


Each row is one student taking one course in one term.

The columns, in groups:

- **Who and what**: `id_student`, `code_module` (the course — `AAA` through
  `GGG`, real names withheld for anonymity), `code_presentation` (which run of
  it: `2013J` started October 2013, `2014B` February 2014).
- **Demographics**: `gender`, `region`, `highest_education`, `imd_band`,
  `age_band`, `disability`.
- **Study context**: `num_of_prev_attempts` (times they'd tried this module
  before), `studied_credits` (course load).
- **The outcome**: `final_result`.

`imd_band` needs a word. IMD is the UK government's **Index of Multiple
Deprivation**, which ranks small geographic areas by relative deprivation.
`0-10%` means the student's home area is among the 10% *most* deprived in the
country; `90-100%` the least. It's a measure of the area, not the person.

In [3]:
print("rows, columns:", student_info.shape)
print()
print(student_info.dtypes)

rows, columns: (32593, 12)

code_module               str
code_presentation         str
id_student              int64
gender                    str
region                    str
highest_education         str
imd_band                  str
age_band                  str
num_of_prev_attempts    int64
studied_credits         int64
disability                str
final_result              str
dtype: object


`object` means text. `int64` means whole numbers.

32,593 rows. That is the number of **enrolments**, and it's worth being careful
about the word — it is not the number of students. More on that shortly.

## Grain — the concept that prevents most data bugs

The **grain** of a table is what one row represents. Getting it wrong is the
most common way to silently corrupt an analysis, because nothing errors. You
just get numbers that are quietly wrong.

For `studentInfo` the grain is *one row per (student, module, presentation)*.
That means the same person can legitimately appear more than once — taking two
courses, or retaking one.

In [4]:
# How many rows share an id_student with some other row?
counts = student_info["id_student"].value_counts()
repeat_students = counts[counts > 1]

print(f"distinct students : {student_info['id_student'].nunique():,}")
print(f"enrolment rows    : {len(student_info):,}")
print(f"students appearing more than once: {len(repeat_students):,}")
print()
example = repeat_students.index[0]
student_info[student_info["id_student"] == example][
    ["id_student", "code_module", "code_presentation", "final_result"]
]

distinct students : 28,785
enrolment rows    : 32,593
students appearing more than once: 3,538



,id_student,code_module,code_presentation,final_result
9587,584077,CCC,2014B,Withdrawn
11525,584077,CCC,2014J,Withdrawn
15596,584077,DDD,2013J,Withdrawn
16828,584077,DDD,2014B,Withdrawn
18139,584077,DDD,2014J,Withdrawn


One person, several courses. Those are **not duplicates** — they are
different enrolments and each has its own outcome.

This single fact has a consequence we'll come back to in notebook 02: when we
later split the data into a part for building the model and a part for testing
it, the same person must not land on both sides.

## Why grain matters: a join that goes wrong

A **join** combines two tables by matching on shared columns — like a lookup.
When the grains match, you get what you expect. When they don't, rows multiply.

Here's the trap, with real tables. `studentVle` is the clickstream: one row per
student *per webpage per day*. Far finer-grained than `studentInfo`.

In [5]:
vle_clicks = load.load_table("studentVle", strict_grain=False)
print(f"studentInfo rows : {len(student_info):,}")
print(f"studentVle rows  : {len(vle_clicks):,}")
vle_clicks.head()

studentInfo rows : 32,593
studentVle rows  : 10,655,280


,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1


In [6]:
# Take one student's enrolment and join the clickstream onto it.
one = student_info[
    (student_info["id_student"] == example)
    & (student_info["code_module"] == student_info[
        student_info["id_student"] == example]["code_module"].iloc[0])
].head(1)

joined = one.merge(
    vle_clicks,
    on=["id_student", "code_module", "code_presentation"],
    how="left",
)

print(f"rows before join : {len(one)}")
print(f"rows after join  : {len(joined):,}")
print()
print("One enrolment became", len(joined), "rows -- one per page per day.")

rows before join : 1
rows after join  : 109

One enrolment became 109 rows -- one per page per day.


One row went in; hundreds came out.

That is *correct* behaviour for a join — but if you had expected one row per
student and then computed, say, an average age, you would now be averaging that
student's age hundreds of times, weighted by how much they clicked. The number
would be wrong and nothing would have warned you.

**The rule:** before joining, know the grain of both sides. If the right-hand
side is finer, you must aggregate it first — collapse it to one row per student
— and only then join.

This project's loader checks grain on every read, so a file that violates its
expected grain fails immediately rather than silently.

## All seven tables

Now the whole set.

In [7]:
tables = load.load_all()

  studentInfo                32,593 rows x 12 cols
  studentRegistration        32,593 rows x  5 cols


  studentVle             10,655,280 rows x  6 cols
  vle                         6,364 rows x  6 cols
  assessments                   206 rows x  6 cols
  studentAssessment         173,912 rows x  5 cols
  courses                        22 rows x  3 cols


What each one is:

| Table | Grain — one row per… | What it's for |
|---|---|---|
| `studentInfo` | student × module × presentation | Demographics + the outcome |
| `studentRegistration` | student × module × presentation | When they joined and, if they left, when |
| `studentVle` | student × module × presentation × page × day | The clickstream — the behavioural record |
| `vle` | page | What each page *is* (forum, quiz, resource…) |
| `assessments` | assessment | Due date, type, weight |
| `studentAssessment` | assessment × student | Submissions and scores |
| `courses` | module × presentation | How many days long |

Two are lookup tables (`vle`, `courses`) — small, and used to translate codes
into meaning. `studentVle` is 10.6 million rows and is where the signal lives.

One subtlety about `studentAssessment`: **only submissions appear in it.** A
student who never handed anything in has no row at all. So a missing row is not
"we don't know" — it means "did not submit", which is very informative.

## The date convention — the thing to internalise

**There are no calendar dates in this dataset.** Every date column is a whole
number counting days from the start of that presentation.

- `0` = first day of teaching
- `28` = four weeks in
- `-30` = a month *before* teaching started

Negative numbers are not errors. Students register weeks ahead, and some drop
out before teaching begins.

In [8]:
reg = tables["studentRegistration"]
reg[["date_registration", "date_unregistration"]].describe().round(1)

,date_registration,date_unregistration
count,32548.0,10072.0
mean,-69.4,49.8
std,49.3,82.5
min,-322.0,-365.0
25%,-100.0,-2.0
50%,-57.0,27.0
75%,-29.0,109.0
max,167.0,444.0


`date_registration` has a median around −57: most students register about two
months early. The minimum is far more negative still.

## Missing values, and how to read them

In [9]:
rows = []
for name, df in tables.items():
    for col in df.columns:
        pct = df[col].isna().mean() * 100
        if pct > 0:
            rows.append({"table": name, "column": col, "missing_%": round(pct, 2)})

pd.DataFrame(rows).sort_values("missing_%", ascending=False).reset_index(drop=True)

,table,column,missing_%
0,vle,week_from,82.39
1,vle,week_to,82.39
2,studentRegistration,date_unregistration,69.10
3,assessments,date,5.34
4,studentInfo,imd_band,3.41
5,studentRegistration,date_registration,0.14
6,studentAssessment,score,0.10


Now read that table properly, because the top row is a trap.

**`date_unregistration`, 69% missing.** This is *not* missing data. The column
records the day a student dropped out, and it is empty precisely when they never
dropped out. The blank **is** the information. Let's prove it:

In [10]:
check = tables["studentRegistration"].merge(
    student_info[["id_student", "code_module", "code_presentation", "final_result"]],
    on=["id_student", "code_module", "code_presentation"],
)
check["has_unreg_date"] = check["date_unregistration"].notna()

pd.crosstab(check["final_result"], check["has_unreg_date"])

has_unreg_date,False,True
final_result,,
Distinction,3024,0
Fail,7043,9
Pass,12361,0
Withdrawn,93,10063


Look at that table. `Withdrawn` has an unregistration date; nothing else does
(bar nine oddities we'll note later). The correspondence is near-perfect.

Which means: **if you gave this column to a model, you would be handing it the
answer.** Anything predicting "withdrew" from "has a withdrawal date" is not
predicting, it's reading. This is the first of several traps, covered properly
in `docs/02-leakage.md`.

The other missing-value rows are more ordinary:

- **`vle.week_from` / `week_to`, 82%** — most pages aren't tied to a teaching week.
- **`assessments.date`, 5.3%** — mostly final exams with no fixed date.
- **`imd_band`, 3.4%** — genuinely missing. This one needs a decision later, and
  it's awkward: the categories have an *order* (`0-10%` < `10-20%` < …), so we
  can't just add a "missing" category without breaking that ordering.
- **`score`, `date_registration`** — a rounding error's worth. Ignorable.

## What to take away

1. **A row in our analysis is an enrolment**, not a student. 32,593 enrolments,
   28,785 people.
2. **Grain is the thing to check before every join.** Mismatched grain multiplies
   rows silently.
3. **All dates are days from course start.** Negatives are pre-course and normal.
4. **Missing doesn't always mean unknown.** In `date_unregistration` it means
   "never withdrew", which makes it the answer in disguise.

Next: [`02-the-target.ipynb`](02-the-target.ipynb) — what exactly are we
predicting, and how common is it?